# Guided Lab: Deep Learning

**Goal:** Build an intuition for how a deep learning model is defined, trained, and debugged, by training a small neural network to recognise handwritten digits.

Throughout the notebook you will find **Task** boxes. They are short (1-3 minutes each), do them as you go.

---
## Part 0: Setup

We use **PyTorch**, the most common framework for learning deep learning. It makes the training loop explicit, which is exactly what we want to understand.

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset, TensorDataset
from torchvision import datasets, transforms

import numpy as np
import matplotlib.pyplot as plt

# Reproducibility: fix the random seed so results are comparable across runs.
torch.manual_seed(0)
np.random.seed(0)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

---
## Part 1: Load and visualise MNIST

**MNIST** is a classic dataset of 70,000 grayscale images (28x28 pixels) of handwritten digits 0-9. It is the "hello world" of deep learning.

The cell below downloads it (once, ~10 MB) and converts each image into a tensor of pixel values scaled to the range `[0, 1]`.

In [ ]:
# transforms.ToTensor() converts a 28x28 image into a [1, 28, 28] float tensor in [0, 1].
transform = transforms.ToTensor()

full_train = datasets.MNIST(root="./data", train=True,  download=True, transform=transform)
full_test  = datasets.MNIST(root="./data", train=False, download=True, transform=transform)

print("Training images:", len(full_train))
print("Test images:    ", len(full_test))

# Look at one example: an (image, label) pair.
image, label = full_train[0]
print("\nOne example:")
print("  image tensor shape:", tuple(image.shape))   # (channels, height, width)
print("  pixel value range: ", float(image.min()), "to", float(image.max()))
print("  label:             ", label)

In [ ]:
# Plot a grid of the first 10 images with their labels.
fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for i, ax in enumerate(axes.flat):
    img, lbl = full_train[i]
    ax.imshow(img.squeeze(), cmap="gray")   # squeeze() drops the channel dim: [1,28,28] -> [28,28]
    ax.set_title(f"label: {lbl}")
    ax.axis("off")
plt.suptitle("MNIST samples")
plt.tight_layout()
plt.show()

In [ ]:
# How balanced are the classes? Count how many images exist per digit.
counts = torch.bincount(full_train.targets)
plt.bar(range(10), counts.numpy())
plt.xticks(range(10))
plt.xlabel("digit")
plt.ylabel("number of images")
plt.title("Class distribution in the training set")
plt.show()

**Task 1.1** A neural network only sees numbers, not pictures. Change the index below and run the cell to inspect the raw pixel matrix of a single digit. Can you "see" the digit in the numbers?

In [ ]:
index = 1  # <-- TODO: try a few different indices (e.g. 1, 7, 42)

img, lbl = full_train[index]
print("Label:", lbl)
# Print rounded pixel values; brighter pixels (closer to 1) form the digit's strokes.
with np.printoptions(precision=1, linewidth=140, suppress=True):
    print(img.squeeze().numpy())

---
## Part 2: How is a deep learning model defined?

A neural network is a stack of **layers**. Each layer applies a linear transformation (`weights @ input + bias`) followed by a non-linear **activation function**. Stacking these lets the network learn complex patterns.

Our model is a simple **Multi-Layer Perceptron (MLP)**:

- **Input:** 784 numbers (28x28 pixels flattened into a vector)
- **Hidden layer:** 128 neurons with a ReLU activation
- **Output:** one score (or *logit*) per class — but how many classes? You will work that out in Task 2.1.

In PyTorch, `nn.Sequential` lets us define a model as a simple ordered stack of layers: data flows straight through them, top to bottom. We wrap it in a small `make_model` helper so we can easily change the hidden size later.

**Task 2.1** The last layer's size is left blank (`NUM_CLASSES = ...`) in the cell below. Before filling it in, answer for yourself:

1. **What** is this model predicting for a given image?
2. **How many** numbers should the output layer produce, and why?

Then set `NUM_CLASSES` to the correct value and run the cell. (If you pick the wrong number, training later will fail with a shape error — a useful clue!)

In [ ]:
NUM_CLASSES = ...  # TODO (Task 2.1): how many values should the output layer produce?

def make_model(hidden_size=128):
    return nn.Sequential(
        nn.Flatten(),                  # [batch, 1, 28, 28] -> [batch, 784]
        nn.Linear(28 * 28, hidden_size),  # input layer -> hidden layer
        nn.ReLU(),                     # activation: turns negatives into 0, keeps positives
        nn.Linear(hidden_size, NUM_CLASSES),  # hidden layer -> one score per class
    )


model = make_model().to(DEVICE)
print(model)

<details><summary>Answer & explanation (open after you've tried Task 2.1)</summary>

**What are we predicting?** For each image, the model predicts **which digit it shows**. The possible answers are the digits `0, 1, 2, ..., 9` — that is **10 mutually exclusive classes**. This is a *multi-class classification* problem.

**So `NUM_CLASSES = 10`.** The output layer produces one number (a *logit*) per class: 10 scores in total. A higher score means the model thinks that class is more likely. To turn the 10 scores into a single prediction we take the index of the largest one (`argmax`), and to turn them into probabilities we apply `softmax`.

**Why exactly 10 and not fewer/more?** The output size must match the number of categories you want to distinguish. Too few and some digits could never be predicted; too many and you have dead outputs that never correspond to a real label. The number of output neurons is dictated by the *task*, not by tuning — unlike the hidden size, which you are free to choose.

</details>

In [ ]:
# How many learnable parameters does this tiny model have?
total = sum(p.numel() for p in model.parameters())
print(f"Total trainable parameters: {total:,}")

# Sanity check: feed one image through the untrained model.
img, _ = full_train[0]
logits = model(img.unsqueeze(0).to(DEVICE))   # unsqueeze adds the batch dimension
print("Output shape:", tuple(logits.shape), "(one score per class)")
print("Raw scores:", logits.detach().cpu().numpy().round(2))

**Task 2.2** The model above has 128 hidden neurons. In the cell below, build a *bigger* model with 256 hidden neurons and print its parameter count. How much did the number of parameters grow?

In [ ]:
# TODO: create a model with hidden_size=256 and print its total parameter count.
big_model = ...
# total_big = sum(p.numel() for p in big_model.parameters())
# print(f"Bigger model parameters: {total_big:,}")

---
## Part 3: What is a loss function?

A freshly created model makes random guesses. To improve it, we need a single number that says **how wrong** the model currently is. That number is the **loss**. Training = adjusting the weights to make the loss as small as possible.

For classification we use **cross-entropy loss**. It compares the model's predicted probabilities against the true label:

- Confident **and correct** -> small loss
- Confident **and wrong** -> large loss
- Unsure (random guessing) -> medium loss

For 10 equally-likely classes, a model that guesses randomly has a loss of about `ln(10) ≈ 2.30`. That is our "knowing nothing" baseline.

In [ ]:
loss_fn = nn.CrossEntropyLoss()

# Imagine the true label is class 3.
true_label = torch.tensor([3])

# Case A: random/uniform scores (the model has no idea).
uniform_logits = torch.zeros(1, 10)
# Case B: confident and CORRECT (high score on class 3).
correct_logits = torch.tensor([[0., 0., 0., 9., 0., 0., 0., 0., 0., 0.]])
# Case C: confident and WRONG (high score on class 8).
wrong_logits = torch.tensor([[0., 0., 0., 0., 0., 0., 0., 0., 9., 0.]])

print(f"Random guess loss:        {loss_fn(uniform_logits, true_label):.3f}  (~ln(10) = 2.30)")
print(f"Confident + correct loss: {loss_fn(correct_logits, true_label):.3f}")
print(f"Confident + wrong loss:   {loss_fn(wrong_logits, true_label):.3f}")

**Task 3.1** Before running the next part: our untrained model knows nothing. What loss value do you *expect* to see on the first training step? Write your guess down, then check it against the first printed loss in Part 4.

---
## Part 4: How is a model trained in code?

Training repeats one core loop. For each **batch** of images:

1. **Forward pass** — feed images through the model to get predictions.
2. **Compute loss** — measure how wrong the predictions are.
3. **Backward pass** (`loss.backward()`) — compute the gradient: which direction each weight should move to reduce the loss.
4. **Optimiser step** (`optimizer.step()`) — nudge the weights in that direction.
5. **Zero the gradients** (`optimizer.zero_grad()`) — reset before the next batch.

One full pass over the whole dataset is called an **epoch**.

To keep this lab fast on a CPU, we train on a **subset** of MNIST (8,000 train / 2,000 test images). The same code works on the full dataset, it just takes longer.

In [ ]:
# Build smaller, CPU-friendly subsets so experiments run in seconds.
train_ds = Subset(full_train, range(8000))
test_ds  = Subset(full_test,  range(2000))

# DataLoaders feed the model shuffled batches of data.
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
test_loader  = DataLoader(test_ds,  batch_size=256, shuffle=False)

print("Train batches per epoch:", len(train_loader))

In [ ]:
def train_one_epoch(model, loader, loss_fn, optimizer):
    '''Run one full pass over the data and return the average loss.'''
    model.train()
    running_loss = 0.0
    for images, labels in loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)

        optimizer.zero_grad()           # 5. reset gradients from the previous step
        logits = model(images)          # 1. forward pass
        loss = loss_fn(logits, labels)  # 2. compute loss
        loss.backward()                 # 3. backward pass (compute gradients)
        optimizer.step()                # 4. update the weights

        running_loss += loss.item() * images.size(0)
    return running_loss / len(loader.dataset)


@torch.no_grad()
def evaluate(model, loader, loss_fn):
    '''Compute average loss and accuracy without updating the model.'''
    model.eval()
    total_loss, correct = 0.0, 0
    for images, labels in loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        logits = model(images)
        total_loss += loss_fn(logits, labels).item() * images.size(0)
        correct += (logits.argmax(dim=1) == labels).sum().item()
    n = len(loader.dataset)
    return total_loss / n, correct / n

In [ ]:
# Put it all together and train for a few epochs.
model = make_model(hidden_size=128).to(DEVICE)
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)

EPOCHS = 8
history = {"train_loss": [], "test_loss": [], "test_acc": []}

for epoch in range(1, EPOCHS + 1):
    train_loss = train_one_epoch(model, train_loader, loss_fn, optimizer)
    test_loss, test_acc = evaluate(model, test_loader, loss_fn)

    history["train_loss"].append(train_loss)
    history["test_loss"].append(test_loss)
    history["test_acc"].append(test_acc)
    print(f"Epoch {epoch:2d} | train loss {train_loss:.3f} | test loss {test_loss:.3f} | test acc {test_acc:.1%}")

**Task 4.1** Look at the first epoch's training loss. Is it close to the `~2.30` baseline you predicted in Task 3.1? By the last epoch, how much has the loss dropped?

---
## Part 5: Visualise training

Numbers in a log are hard to read. **Learning curves** make the story obvious: loss should go *down*, accuracy should go *up*, and the train and test curves should stay close together.

In [ ]:
epochs_axis = range(1, EPOCHS + 1)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(epochs_axis, history["train_loss"], "o-", label="train")
ax1.plot(epochs_axis, history["test_loss"],  "o-", label="test")
ax1.set_xlabel("epoch"); ax1.set_ylabel("loss"); ax1.set_title("Loss over time")
ax1.legend()

ax2.plot(epochs_axis, history["test_acc"], "o-", color="green")
ax2.set_xlabel("epoch"); ax2.set_ylabel("accuracy"); ax2.set_title("Test accuracy over time")
ax2.set_ylim(0, 1)

plt.tight_layout()
plt.show()

In [ ]:
# See the model in action: predictions on a few test images (green = correct, red = wrong).
model.eval()
images, labels = next(iter(test_loader))
with torch.no_grad():
    preds = model(images.to(DEVICE)).argmax(dim=1).cpu()

fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for i, ax in enumerate(axes.flat):
    ax.imshow(images[i].squeeze(), cmap="gray")
    correct = preds[i].item() == labels[i].item()
    ax.set_title(f"pred {preds[i].item()} / true {labels[i].item()}",
                 color="green" if correct else "red")
    ax.axis("off")
plt.suptitle("Model predictions on test images")
plt.tight_layout()
plt.show()

**Task 5.1** Look at the loss plot. Are the train and test curves close together, or is there a growing gap? A growing gap is an early sign of **overfitting** (Part 7).

---
## Part 6: Training hyperparameters

**Parameters** are the weights the model *learns*. **Hyperparameters** are the knobs *you* set before training. The most important ones:

- **Learning rate** — how big each weight update is. The single most important knob.
- **Batch size** — how many images are processed before each update.
- **Epochs** — how many times we pass over the full dataset.
- **Hidden size** — model capacity (how many neurons).

Let's wrap training in a reusable function so we can experiment quickly.

In [ ]:
def run_experiment(lr=0.1, hidden_size=128, batch_size=64, epochs=8, train_subset=8000, verbose=False):
    '''Train a fresh model with the given hyperparameters and return the history dict.'''
    torch.manual_seed(0)
    ds = Subset(full_train, range(train_subset))
    loader = DataLoader(ds, batch_size=batch_size, shuffle=True)

    model = make_model(hidden_size=hidden_size).to(DEVICE)
    loss_fn = nn.CrossEntropyLoss()
    optimizer = torch.optim.SGD(model.parameters(), lr=lr)

    hist = {"train_loss": [], "test_loss": [], "test_acc": []}
    for epoch in range(1, epochs + 1):
        tr = train_one_epoch(model, loader, loss_fn, optimizer)
        te, acc = evaluate(model, test_loader, loss_fn)
        hist["train_loss"].append(tr)
        hist["test_loss"].append(te)
        hist["test_acc"].append(acc)
        if verbose:
            print(f"  epoch {epoch:2d} | train {tr:.3f} | test {te:.3f} | acc {acc:.1%}")
    return hist

In [ ]:
# Compare three learning rates. Watch how dramatically this single knob changes learning.
for lr in [0.01, 0.1, 1.0]:
    hist = run_experiment(lr=lr, epochs=8)
    plt.plot(range(1, 9), hist["train_loss"], "o-", label=f"lr = {lr}")

plt.xlabel("epoch"); plt.ylabel("train loss")
plt.title("Effect of the learning rate")
plt.legend()
plt.show()

**Task 6.1** Run your own experiment below. Try changing one hyperparameter at a time and note the final test accuracy. Some ideas:

- A very small learning rate (e.g. `0.001`) — does it learn too slowly?
- A larger hidden size (e.g. `256`)
- A larger batch size (e.g. `256`)

In [ ]:
# TODO: change these values and observe the final test accuracy.
hist = run_experiment(lr=0.1, hidden_size=128, batch_size=64, epochs=8, verbose=True)
print(f"\nFinal test accuracy: {hist['test_acc'][-1]:.1%}")

---
## Part 7: Failure modes

Models rarely fail with an error message, they fail by learning poorly. Recognising the *shape* of a failure is a core skill. We will reproduce the three most common ones.

### 7a. Learning rate too high — training diverges
If the learning rate is too large, weight updates overshoot the target. The loss bounces around or explodes to `NaN` instead of decreasing.

In [ ]:
hist = run_experiment(lr=8.0, epochs=8, verbose=True)
plt.plot(range(1, 9), hist["train_loss"], "o-", color="red")
plt.xlabel("epoch"); plt.ylabel("train loss")
plt.title("Failure mode: learning rate too high (loss does not settle)")
plt.show()

### 7b. Overfitting — memorising instead of generalising
With too little data (or too long training), the model memorises the training set. Training loss keeps dropping while test loss climbs back up: the gap between the two curves is the tell-tale sign.

In [ ]:
# Train on only 200 images for many epochs to force overfitting.
hist = run_experiment(lr=0.1, epochs=40, train_subset=200)

plt.plot(range(1, 41), hist["train_loss"], label="train loss")
plt.plot(range(1, 41), hist["test_loss"],  label="test loss")
plt.xlabel("epoch"); plt.ylabel("loss")
plt.title("Failure mode: overfitting (train keeps dropping, test rises)")
plt.legend()
plt.show()
print(f"Final train loss: {hist['train_loss'][-1]:.3f} | final test loss: {hist['test_loss'][-1]:.3f}")

### 7c. Underfitting — learning rate too small / not enough training
The opposite problem: the model never learns enough. The loss decreases painfully slowly and accuracy stays low. Below, a tiny learning rate barely moves the loss in the time we give it.

In [ ]:
hist = run_experiment(lr=0.0005, epochs=8, verbose=True)
print(f"\nFinal test accuracy: {hist['test_acc'][-1]:.1%}  (compare to ~90%+ with lr=0.1)")

**Task 7.1** You are given a model whose **training accuracy is 99% but test accuracy is 70%**. Which failure mode is this? Name two things you could try to fix it.

<details><summary>Show answer</summary>

This is **overfitting**. Fixes include: use more training data, train for fewer epochs (early stopping), reduce model size, or add regularisation (e.g. dropout / weight decay).

</details>

---
## Wrap-up

You have completed a full deep learning workflow:

- Loaded and visualised **MNIST**
- Defined a neural network as an `nn.Module` (layers, activations, parameters)
- Used **cross-entropy loss** to measure "how wrong" the model is
- Written the **training loop**: forward -> loss -> backward -> step
- Read **learning curves** to track progress
- Tuned **hyperparameters**, especially the learning rate
- Recognised **divergence, overfitting, and underfitting**

### Going further
- Swap the MLP for a **convolutional network (CNN)** — the standard for images.
- Train on the **full** MNIST dataset and push past 97% accuracy.
- Replace `SGD` with the **Adam** optimiser (`torch.optim.Adam`) and compare.
- Add **dropout** (`nn.Dropout`) and see its effect on the overfitting experiment.